<h2> Aniqa's Notebook for SQL portion</h2>

Guiding Question: Which airlines have the fastest service (quickness of getting on and off the air, maybe lack of delays)?

In [2]:
import pandas as pd
import pymysql
import getpass # May need to import as `from getpass import getpass`
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import sqlite3

In [3]:
conn = sqlite3.connect("flights.db")

In [4]:
airlines = pd.read_csv("airlines.csv")
airports = pd.read_csv("airports.csv")
flights = pd.read_csv("flights.csv", low_memory=False)

In [ ]:
airlines.to_sql("airlines", conn, if_exists="replace", index=False)
airports.to_sql("airports", conn, if_exists="replace", index=False)
flights.to_sql("flights", conn, if_exists="replace", index=False) #big dataset, takes a while to load

5819079

<h3> Running Basic Queries </h3>
To figure out schema And foreign/primary keys for joins 

In [6]:
pd.read_sql("SELECT * FROM airlines;", conn) #primary_key is IATA_CODE

,IATA_CODE,AIRLINE
0,UA,United Air Lines Inc.
1,AA,American Airlines Inc.
2,US,US Airways Inc.
3,F9,Frontier Airlines Inc.
4,B6,JetBlue Airways
5,OO,Skywest Airlines Inc.
6,AS,Alaska Airlines Inc.
7,NK,Spirit Air Lines
8,WN,Southwest Airlines Co.
9,DL,Delta Air Lines Inc.


In [7]:
pd.read_sql("SELECT * FROM airports;", conn) #primary_key is IATA_CODE

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919
3,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183
4,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447
...,...,...,...,...,...,...,...
317,WRG,Wrangell Airport,Wrangell,AK,USA,56.48433,-132.36982
318,WYS,Westerly State Airport,West Yellowstone,MT,USA,44.68840,-111.11764
319,XNA,Northwest Arkansas Regional Airport,Fayetteville/Springdale/Rogers,AR,USA,36.28187,-94.30681
320,YAK,Yakutat Airport,Yakutat,AK,USA,59.50336,-139.66023


In [16]:
pd.read_sql("SELECT * FROM flights LIMIT 5;", conn) #primary_key is flight number, 
#foreign key is AIRLINE, which references IATA_CODE in airlines
#foreign key is ORIGIN_AIRPORT, DESTINATION_AIRPORT and it references IATA_CODE in 


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,None,None,None,None,None,None
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,None,None,None,None,None,None
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,None,None,None,None,None,None
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,None,None,None,None,None,None
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,None,None,None,None,None,None


<h3>Query to see all columns of **flights** table</h3>

In [9]:
pd.read_sql_query("""
PRAGMA table_info(flights);
""", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,YEAR,INTEGER,0,None,0
1,1,MONTH,INTEGER,0,None,0
2,2,DAY,INTEGER,0,None,0
3,3,DAY_OF_WEEK,INTEGER,0,None,0
4,4,AIRLINE,TEXT,0,None,0
5,5,FLIGHT_NUMBER,INTEGER,0,None,0
6,6,TAIL_NUMBER,TEXT,0,None,0
7,7,ORIGIN_AIRPORT,TEXT,0,None,0
8,8,DESTINATION_AIRPORT,TEXT,0,None,0
9,9,SCHEDULED_DEPARTURE,INTEGER,0,None,0


<h4> Main Queries to Answer Guiding Question

IMPORTANT: negative values in DEPARTURE_DELAYS and ARRIVAL_DELAYS means the flight arrived **early**
positive values means it's a delay

<h5> Analyzing Departures

In [ ]:
#this query is done to successfully carry out the join between flights and airlines tables
pd.read_sql_query('''SELECT 
            a.AIRLINE, f.DEPARTURE_TIME, f.DEPARTURE_DELAY
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE DEPARTURE_DELAY IS NOT NULL
            ORDER BY CAST(DEPARTURE_DELAY AS INTEGER)
            LIMIT 10;''', conn)

,AIRLINE,DEPARTURE_TIME,DEPARTURE_DELAY
0,Alaska Airlines Inc.,1553.0,-82.0
1,American Airlines Inc.,2042.0,-68.0
2,Delta Air Lines Inc.,559.0,-61.0
3,Skywest Airlines Inc.,921.0,-56.0
4,Atlantic Southeast Airlines,1310.0,-55.0
5,Delta Air Lines Inc.,1817.0,-52.0
6,Skywest Airlines Inc.,647.0,-48.0
7,Alaska Airlines Inc.,1742.0,-48.0
8,Alaska Airlines Inc.,1743.0,-47.0
9,Frontier Airlines Inc.,644.0,-46.0


In [24]:
#query is looking at average delay (in minutes) for each airline, sorted in ascending order
pd.read_sql_query('''SELECT 
            a.AIRLINE, AVG(f.DEPARTURE_DELAY) AS Average_Delay
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE DEPARTURE_DELAY IS NOT NULL
            GROUP BY a.AIRLINE
            ORDER BY AVG(f.DEPARTURE_DELAY) ASC
            ;''', conn)

,AIRLINE,Average_Delay
0,Hawaiian Airlines Inc.,0.485713
1,Alaska Airlines Inc.,1.785801
2,US Airways Inc.,6.141137
3,Delta Air Lines Inc.,7.369254
4,Skywest Airlines Inc.,7.801104
5,Atlantic Southeast Airlines,8.715934
6,American Airlines Inc.,8.900856
7,Virgin America,9.022595
8,American Eagle Airlines Inc.,10.125188
9,Southwest Airlines Co.,10.581986


In [ ]:
#query is looking at minimum delay (in minutes) for each airline, sorted in ascending order
#minimum for each airline means the earliest an airline has landed before its scheduled time
#ex: -60 minutes means the flight arrived 60 minutes EARLIER than scheduled
#ascending order is appropriate here since we are dealing with negative values
pd.read_sql_query('''SELECT 
            a.AIRLINE, MIN(f.DEPARTURE_DELAY)
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE DEPARTURE_DELAY IS NOT NULL
            GROUP BY a.AIRLINE
            ORDER BY MIN(f.DEPARTURE_DELAY) ASC
            ;''', conn)

,AIRLINE,MIN(f.DEPARTURE_DELAY)
0,Alaska Airlines Inc.,-82.0
1,American Airlines Inc.,-68.0
2,Delta Air Lines Inc.,-61.0
3,Skywest Airlines Inc.,-56.0
4,Atlantic Southeast Airlines,-55.0
5,Frontier Airlines Inc.,-46.0
6,United Air Lines Inc.,-40.0
7,Spirit Air Lines,-37.0
8,American Eagle Airlines Inc.,-36.0
9,US Airways Inc.,-35.0


In [22]:
#query is looking at maximum delay (in minutes) for each airline, sorted in descending order
pd.read_sql_query('''SELECT 
            a.AIRLINE, MAX(f.DEPARTURE_DELAY)
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE DEPARTURE_DELAY IS NOT NULL
            GROUP BY a.AIRLINE
            ORDER BY MAX(f.DEPARTURE_DELAY) DESC
            ;''', conn)

,AIRLINE,MAX(f.DEPARTURE_DELAY)
0,American Airlines Inc.,1988.0
1,American Eagle Airlines Inc.,1544.0
2,Hawaiian Airlines Inc.,1433.0
3,Skywest Airlines Inc.,1378.0
4,United Air Lines Inc.,1314.0
5,Delta Air Lines Inc.,1289.0
6,Atlantic Southeast Airlines,1274.0
7,Frontier Airlines Inc.,1112.0
8,JetBlue Airways,1006.0
9,Alaska Airlines Inc.,963.0


<h5> Analyzing Arrivals

In [ ]:
#this query is done to successfully carry out the join between flights and airlines tables
pd.read_sql_query('''SELECT 
            a.AIRLINE, f.ARRIVAL_TIME, f.ARRIVAL_DELAY 
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE ARRIVAL_DELAY IS NOT NULL
            ORDER BY CAST(ARRIVAL_DELAY AS INTEGER)
            LIMIT 10;''', conn)

,AIRLINE,ARRIVAL_TIME,ARRIVAL_DELAY
0,US Airways Inc.,538.0,-87.0
1,American Airlines Inc.,40.0,-87.0
2,Alaska Airlines Inc.,1927.0,-82.0
3,United Air Lines Inc.,2129.0,-81.0
4,Virgin America,1429.0,-81.0
5,American Airlines Inc.,2312.0,-80.0
6,American Airlines Inc.,1837.0,-80.0
7,Alaska Airlines Inc.,1940.0,-80.0
8,Delta Air Lines Inc.,552.0,-79.0
9,Delta Air Lines Inc.,1738.0,-79.0


In [ ]:
#aggregation query to make steps towards answering question
pd.read_sql_query('''SELECT 
            a.AIRLINE, AVG(f.ARRIVAL_DELAY), MIN(f.ARRIVAL_DELAY), MAX(f.ARRIVAL_DELAY)
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE
            WHERE ARRIVAL_DELAY IS NOT NULL
            GROUP BY a.AIRLINE
            ORDER BY AVG(f.ARRIVAL_DELAY) ASC
            LIMIT 10;''', conn)

,AIRLINE,AVG(f.ARRIVAL_DELAY),MIN(f.ARRIVAL_DELAY),MAX(f.ARRIVAL_DELAY)
0,Alaska Airlines Inc.,-0.976563,-82.0,950.0
1,Delta Air Lines Inc.,0.186754,-79.0,1274.0
2,Hawaiian Airlines Inc.,2.023093,-67.0,1467.0
3,American Airlines Inc.,3.451372,-87.0,1971.0
4,US Airways Inc.,3.706209,-87.0,750.0
5,Southwest Airlines Co.,4.374964,-73.0,659.0
6,Virgin America,4.737706,-81.0,651.0
7,United Air Lines Inc.,5.431594,-81.0,1294.0
8,Skywest Airlines Inc.,5.845652,-69.0,1372.0
9,American Eagle Airlines Inc.,6.457873,-63.0,1528.0
